In [ ]:
from google.colab import files
uploaded = files.upload()

# Verify uploads:
import os
for name in uploaded:
  print(f"  Uploaded: {name} ({os.path.getsize(name)} bytes)")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score

# 1. LOAD DATA

A_path = "annotations_A.csv"
B_path = "annotations_B.csv"
C_path = "annotations_C (1).csv"

a_raw = pd.read_csv(A_path)
b_raw = pd.read_csv(B_path)
c_raw = pd.read_csv(C_path)

print("=== RAW DATA SUMMARY ===")
print(f"A: {len(a_raw)} rows | {a_raw['annotation'].value_counts().to_dict()}")
print(f"B: {len(b_raw)} rows | {b_raw['annotation'].value_counts().to_dict()}")
print(f"C: {len(c_raw)} rows | {c_raw['annotation'].value_counts().to_dict()}")


# 2. MERGE ON pair_id
a = a_raw[['pair_id', 'original', 'orig_label', 'edit', 'annotation']].rename(columns={'annotation': 'A'})
b = b_raw[['pair_id', 'annotation']].rename(columns={'annotation': 'B'})
c = c_raw[['pair_id', 'annotation']].rename(columns={'annotation': 'C'})

df = a.merge(b, on='pair_id').merge(c, on='pair_id').sort_values('pair_id').reset_index(drop=True)

assert len(df) == 200, f"Expected 200 common pairs, got {len(df)}"
assert df['pair_id'].nunique() == 200, "Duplicate pair_ids detected"
print(f"\nMerged: {len(df)} pairs, all 3 annotators present for every pair ✓")

# Binary encoding: FLIP=1, PRESERVE=0
for col in ['A', 'B', 'C']:
    df[col + '_bin'] = (df[col] == 'FLIP').astype(int)


# 3. PAIRWISE COHEN'S KAPPA
print("\n" + "=" * 60)
print("PAIRWISE COHEN'S KAPPA")
print("=" * 60)

annotator_pairs = [('A', 'B'), ('A', 'C'), ('B', 'C')]
kappas = []
for a_name, b_name in annotator_pairs:
    k = cohen_kappa_score(df[a_name + '_bin'], df[b_name + '_bin'])
    agree = (df[a_name + '_bin'] == df[b_name + '_bin']).mean()
    kappas.append(k)
    print(f"  {a_name:10s} vs {b_name:10s}: κ = {k:.4f}  (raw agreement = {agree:.1%})")

mean_kappa = np.mean(kappas)
print(f"\n  Mean pairwise κ = {mean_kappa:.4f}")


# 4. FLEISS' KAPPA (3 raters, 2 categories)
print("\n" + "=" * 60)
print("FLEISS' KAPPA (3 raters, 2 categories)")
print("=" * 60)

N = len(df)   # number of subjects (pairs)
n = 3         # number of raters
k_cats = 2    # number of categories (FLIP, PRESERVE)

mat = np.zeros((N, k_cats), dtype=int)
for i in range(N):
    for col in ['A_bin', 'B_bin', 'C_bin']:
        mat[i, df.loc[i, col]] += 1

# P_i: proportion of agreeing rater pairs for each subject
P_i = (np.sum(mat ** 2, axis=1) - n) / (n * (n - 1))
P_bar = np.mean(P_i)

# P_e: expected agreement by chance
p_j = np.sum(mat, axis=0) / (N * n)
P_e = np.sum(p_j ** 2)

fleiss_kappa = (P_bar - P_e) / (1 - P_e)

print(f"  P̄ (observed agreement)  = {P_bar:.4f}")
print(f"  P_e (chance agreement)   = {P_e:.4f}")
print(f"  Fleiss' κ                = {fleiss_kappa:.4f}")

# Interpretation (Landis & Koch, 1977)
if fleiss_kappa >= 0.81:
    interp = "almost perfect"
elif fleiss_kappa >= 0.61:
    interp = "substantial"
elif fleiss_kappa >= 0.41:
    interp = "moderate"
elif fleiss_kappa >= 0.21:
    interp = "fair"
else:
    interp = "slight"
print(f"  Interpretation: '{interp}' (Landis & Koch, 1977)")

# 5. AGREEMENT BREAKDOWN
print("\n" + "=" * 60)
print("AGREEMENT BREAKDOWN")
print("=" * 60)

# Majority vote
df['majority'] = df[['A_bin', 'B_bin', 'C_bin']].sum(axis=1).apply(
    lambda x: 'FLIP' if x >= 2 else 'PRESERVE'
)
df['unanimous'] = df[['A_bin', 'B_bin', 'C_bin']].apply(
    lambda r: r.nunique() == 1, axis=1
)

n_unanimous = df['unanimous'].sum()
n_majority_flip = (df['majority'] == 'FLIP').sum()
n_majority_preserve = (df['majority'] == 'PRESERVE').sum()

print(f"  Unanimous agreement:    {n_unanimous}/{N} ({n_unanimous/N:.1%})")
print(f"  Majority-vote FLIP:     {n_majority_flip}")
print(f"  Majority-vote PRESERVE: {n_majority_preserve}")

# Per-category unanimity
flip_mask = df['majority'] == 'FLIP'
pres_mask = df['majority'] == 'PRESERVE'
flip_unan = df.loc[flip_mask, 'unanimous'].mean()
pres_unan = df.loc[pres_mask, 'unanimous'].mean()
print(f"\n  Unanimity among FLIP pairs ({flip_mask.sum()}):     {flip_unan:.1%}")
print(f"  Unanimity among PRESERVE pairs ({pres_mask.sum()}): {pres_unan:.1%}")

# 6. DISAGREEMENT CASES
print("\n" + "=" * 60)
print("DISAGREEMENT CASES")
print("=" * 60)

disagree = df[~df['unanimous']].sort_values('pair_id')
print(f"  Total: {len(disagree)} pairs")
print(f"  All disagreements on FLIP pairs: {(disagree['majority']=='FLIP').all()}\n")
for _, row in disagree.iterrows():
    print(f"  Pair {row['pair_id']} (orig_label: {row['orig_label']})")
    print(f"    Original: {row['original']}")
    print(f"    Edit:     {row['edit']}")
    print(f"    A: {row['A']}, B: {row['B']}, C: {row['C']}")
    print(f"    → Majority: {row['majority']}")
    print()

# 7. GROUND-TRUTH ALIGNMENT CHECK
print("=" * 60)
print("GROUND-TRUTH ALIGNMENT CHECK")
print("  (pair_id 0-99 = intended FLIP, 100-199 = intended PRESERVE)")
print("=" * 60)

df['intended'] = df['pair_id'].apply(lambda x: 'FLIP' if x < 100 else 'PRESERVE')
mismatches = df[df['majority'] != df['intended']]

print(f"  Majority vs intended mismatches: {len(mismatches)}")
if len(mismatches) > 0:
    for _, row in mismatches.iterrows():
        print(f"    Pair {row['pair_id']}: intended={row['intended']}, majority={row['majority']}")
else:
    print("  All majority votes match intended labels ✓")

print(f"\n  Dataset design:     100 FLIP / 100 PRESERVE")
print(f"  Annotator majority: {n_majority_flip} FLIP / {n_majority_preserve} PRESERVE")

# 8. SUMMARY TABLE
avg_raw = np.mean([(df[a+'_bin']==df[b+'_bin']).mean() for a,b in annotator_pairs])
print("\n" + "=" * 60)
print("SUMMARY FOR PAPER")
print("=" * 60)
print(f"  Fleiss' κ                    = {fleiss_kappa:.3f}")
print(f"  Mean pairwise Cohen's κ      = {mean_kappa:.3f}")
print(f"  Mean raw pairwise agreement  = {avg_raw:.1%}")
print(f"  Unanimous agreement          = {n_unanimous}/200 ({n_unanimous/200:.1%})")
print(f"  Disagreement cases           = {N - n_unanimous}")
print(f"  All disagreements on FLIP    = {(disagree['majority']=='FLIP').all()}")
print(f"  PRESERVE pair unanimity      = {pres_unan:.1%}")
print(f"  FLIP pair unanimity          = {flip_unan:.1%}")
print(f"  Ground-truth mismatches      = {len(mismatches)}")
print(f"  Interpretation               = {interp}")